In [1]:
import warnings
warnings.filterwarnings('ignore')

# LangChain의 주요 RAG 컴포넌트 소개

# 라이브러리 설치

## 설치하는 라이브러리의 역할

설치하는 4개의 라이브러리는 RAG 시스템 구축의 각 단계를 담당한다.

`langchain_openai`: 모델 연결  
OpenAI의 GPT 모델(LLM)과 Embedding 모델을 LangChain에서 쉽게 쓸 수 있게한다. ChatOpenAI, OpenAIEmbeddings 클래스 등을 포함한다.  

`langchain_community`: 확장 도구  
다양한 로더(WebBaseLoader), 벡터 스토어, 툴들이 모여있고 여러 외부 서비스(Wikipedia, Google Search 등)와의 연결고리 역할을 한다.

`langchain_chroma`: 데이터 저장소  
벡터 데이터베이스인 Chroma를 사용하기 위한 전용 라이브러리로 데이터를 메모리나 로컬에 저장하고 검색하는 엔진이다.

`gradio`: 사용자 인터페이스  
파이썬 코드로만 웹 채팅 화면을 만들어주는 라이브러리로 RAG 완성한 후, 실제로 질문을 입력하고 답변을 받는 `챗봇`을 띄울 때 사용한다.

In [2]:
# !pip install langchain_openai langchain_community langchain_chroma gradio

# 환경 설정

## .env 파일로 환경 변수 설정

`python_dotenv`: 파이썬 프로젝트에서 환경 변수를 안전하게 관리할 수 있도록 도와주는 라이브러리이다.

In [3]:
# !pip install python_dotenv

환경 변수를 안전하게 관리할 수 있도록 도와주는 라이브러리를 사용하기 위해 import 한다.

In [4]:
from dotenv import load_dotenv

# load_dotenv() 함수로 .env 파일의 내용을 환경 변수로 로드한다. env 파일은 소스 코드가 작성되는 폴더에 만든다.
load_dotenv()

True

.env 파일의 환경 변수 정보는 아래의 코드를 실행하면 얻어올 수 있다.

`import os`

`api_key = os.getenv('OPENAI_API_KEY')`  
`api_key`

## 기본 라이브러리

파이썬에서 파일 시스템을 다루기 위해 os, glob 라이브러리를 import 한다.  
`os`: 파일의 경로 조작, 디렉터리(폴더) 생성/삭제, 환경 변수 확인 등 운영체제(Operating System, OS)가 하는 일을 파이썬이 할 수 있게 한다.  
`glob`: 특정 패턴과 일치하는 파일이나 경로를 찾아내기 위해 사용한다. 와일드 카드 문자를 사용할 수 있다.

In [5]:
import os
from glob import glob

텍스트 파일 목록 가져오기

특정 폴더 안에 있는 여러 개의 텍스트 파일 중에서 이름이 특정 규칙을 만족하는 파일들을 한 번에 찾아낸다.

In [6]:
# 서로 다른 운영체제(Windows, Mac, Linux 등)에서도 파일 경로가 깨지지 않게 합쳐준다.
# os.path.join('./data', '*_KR.txt')
# glob() 함수는 인수로 지정된 내용을 만족하는 파일 경로를 리스트로 리턴한다.
txt_files = glob(os.path.join('./data', '*_KR.txt'))
txt_files

['./data\\리비안_KR.txt', './data\\테슬라_KR.txt']

# LangChain RAG 구현

## 문서 로더(Document Loader)

텍스트 파일을 읽어들이기 위해서 TextLoader를 import 한다.  
TextLoader는 단순히 글자만 읽는 게 아니라, 내용(page_content)과 정보(metadata)를 함께 묶어서 관리한다.

In [7]:
from langchain_community.document_loaders import TextLoader

TextLoader 클래스로 파일의 경로와 이름, 인코딩 방식을 넘겨서 파일을 읽어온다.

In [8]:
loader = TextLoader(txt_files[0], encoding='utf-8')
# load() 메소드로 TextLoader 객체에서 실제 문서 내용(metadata, page_content)을 얻어와서 Document 객체로 리턴한다.
data = loader.load()
data

[Document(metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다. 2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다. 주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다. 이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다. 리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다. 2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.\n\n리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.\n')]

In [9]:
type(data)

list

In [10]:
data[0].metadata

{'source': './data\\리비안_KR.txt'}

In [11]:
data[0].page_content

'리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다. 2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다. 주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다. 이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다. 리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다. 2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.\n\n리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.\n'

txt_files에 저장된 모든 파일을 읽어와서 연결한다.

파이썬에서 진행 상태바(progress bar)를 만들어주기 위해서 tqdm를 import 한다.

In [12]:
from tqdm import tqdm

In [13]:
data = []

# 리스트를 tqdm으로 감싸면 반복문이 돌 때마다 화면에 진행 상태바가 나타난다.
for txt_file in tqdm(txt_files):
    loader = TextLoader(txt_file, encoding='utf-8')
    data += loader.load()
    
len(data)

100%|██████████| 2/2 [00:00<00:00, 1329.63it/s]


2

In [14]:
type(data[0])

langchain_core.documents.base.Document

In [15]:
print(data[0].page_content)

리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다. 2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다. 주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.

리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다. 이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다. 리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다. 2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.

리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다.



In [16]:
print(data[1].page_content)

테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.

2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다. SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12.9%를 차지했습니다.



## 텍스트 분할기(Text Splitter)

긴 문서를 처리 가능한 크기의 청크로 분할한다.

각 문서의 글자 수를 계산한다.

In [17]:
char_count = [len(doc.page_content) for doc in data]
char_count

[514, 451]

문서 데이터를 검색에 최적화된 작은 조각(chunk)으로 나누기 위해 CharacterTextSplitter를 import 한다.  
사용자가 지정한 특정 문자를 기준으로 텍스트를 끊어준다.

In [18]:
from langchain_text_splitters import CharacterTextSplitter

RAG 시스템에서 이 과정이 중요한 이유는 문서가 너무 길면 질문과 관련된 핵심 내용을 찾기 어렵고 모델의 토큰 제한에 걸리기 때문이다.  
CharacterTextSplitter는 먼저 `\n\n`를 기준으로 문단들을 자르고 그런다음 이 문단들을 조합해서 250자에 최대한 가깝게 묶어 하나의 청크를 만든다.  
CharacterTextSplitter는 단순히 문자 수로만 텍스트를 자르지 않고 문장이나 단락의 경계를 존중하려고 하기때문에 지정한 청크의 최대 길이를 초과할 수 있다.

In [19]:
text_splitter = CharacterTextSplitter(
    chunk_size=250, # 청크의 최대 길이를 250자로 제한한다. 무조건 250자 이내로 청크를 만들지 않는다.
    chunk_overlap=50, # 문맥이 끊기는 것을 방지하기 위해서 청크와 청크 사이에 50자 만큼 겹치는 부분을 둔다.
    separator='\n\n' # 텍스트를 청크로 나눌 때 기준점을 엔터 두 번(문단, 단락)을 기준으로 나눈다.
)

# split_documents() 메소드로 텍스트 파일에서 읽어온 데이터가 저장된 Document 객체를 넘겨서 청크로 나눈다.
texts = text_splitter.split_documents(data)
print(f'생성된 청크 개수: {len(texts)}')
print(f'각각의 청크 길이: {[len(text.page_content) for text in texts]}')
print(f'각각의 청크 길이: {list(len(text.page_content) for text in texts)}')

Created a chunk of size 282, which is longer than the specified 250
Created a chunk of size 268, which is longer than the specified 250


생성된 청크 개수: 5
각각의 청크 길이: [175, 282, 52, 268, 180]
각각의 청크 길이: [175, 282, 52, 268, 180]


`Created a chunk of size 282, which is longer than the specified 250`  
위 메시지는 CharacterTextSplitter가 지정한 글자수(250자) 제한을 넘어서는 282자짜리 청크를 만들어냈을 때 문제가 발생되면 출력되는 메시지 이다.  
이 문제를 해결하는 방법은 CharacterTextSplitter를 RecursiveCharacterTextSplitter로 교체하거나 separator를 문단(`\n\n`)에서 줄바꿈(`\n`)이나 마침표(`.`) 단위로 줄여서 250자를 넘지 않게 만들 수 있다.

In [20]:
print(f'첫 번째 청크의 내용: {texts[0].page_content}')

첫 번째 청크의 내용: 리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다. 2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다. 주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.


In [21]:
print(f'두 번째 청크의 내용: {texts[1].page_content}')

두 번째 청크의 내용: 리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다. 이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다. 리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다. 2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다.


In [22]:
# chunk_overlap에 의해서 겹치는 부분 출력한다.
print(f'첫 번째 청크의 최종 50자: {texts[0].page_content[-50:]}')
print(f'두 번째 청크의 처음 50자: {texts[1].page_content[:50]}')

첫 번째 청크의 최종 50자: 했습니다. 주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.
두 번째 청크의 처음 50자: 리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 


In [ ]:
# 위의 CharacterTextSplitter 설정에서 separator만 ''으로 수정한다.
text_splitter = CharacterTextSplitter(chunk_size=250, chunk_overlap=50, separator='.')

texts = text_splitter.split_documents(data)
print(f'생성된 청크 개수: {len(texts)}')
print(f'각각의 청크 길이: {list(len(text.page_content) for text in texts)}')

In [ ]:
print(f'첫 번째 청크의 최종 50자: {texts[0].page_content[-50:]}')
print(f'두 번째 청크의 처음 50자: {texts[1].page_content[:50]}')

## 임베딩(Embedding)

텍스트를 벡터로 변환하는 모델

<img src="./embeddings1.png" width="800" align="left" />

임베딩 모델의 활용

<img src="./embeddings2.png" width="800" align="left" />

OpenAI의 텍스트 임베딩 모델을 사용할 수 있도록 OpenAIEmbeddings를 import 한다. 임베딩은 텍스트를 숫자 리스트로 바꿔준다.

In [23]:
from langchain_openai import OpenAIEmbeddings

OpenAIEmbeddings의 모델 비교

`모델명                               가격(1M 토큰당)    기본 차원 수(Dimensions)    추천 용도 및 특징                               `  
`text-embedding-3-small(가장 추천)    $0.02              1,536                       가성비 최고 모델. 일반적인 RAG 시스템           `   
`text-embedding-3-large(가장 강력)    $0.13              3,072                       최고 성능 모델.(6.5배 비쌈). 고도의 정밀도 요구 `  
`text-embedding-ada-002(구형)         $0.10              1,536                       기존 레거시 모델. 성능이 낮고 가격도 5배 비싸다.`

In [24]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

sample_text = '테슬라 창업자는 누구인가요?'
# embed_query() 메소드로 임베딩할 청크를 넘겨서 숫자로 벡터화 한다.
vector = embeddings.embed_query(sample_text)
print(f'text-embedding-3-small 모델의 임베딩 차원: {len(vector)}')

text-embedding-3-small 모델의 임베딩 차원: 1536


## 벡터 저장소

임베딩된 벡터를 저장하고 검색까지 가능한 데이터베이스

기본 개념  
&nbsp;&nbsp;&nbsp;&nbsp;▶ 비정형 데이터(텍스트 등)를 벡터로 임베딩하여 저장한다.  
&nbsp;&nbsp;&nbsp;&nbsp;▶ 입력된 쿼리를 벡터로 임베딩하여, 가장 '유사한' 임베딩 벡터를 검색한다.

<img src="./vectorStore1.png" width="700" align="left" />

Chroma, FAISS, Pinecone 등 다양한 옵션을 제공한다.

<img src="./vectorStore2.png" width="700" align="left" />

`Chroma`  
&nbsp;&nbsp;&nbsp;&nbsp;▶ 사용자 편의성이 우수한 오픈소스 벡터 저장소  
&nbsp;&nbsp;&nbsp;&nbsp;▶ langchain_chroma 라이브러리로 설치 가능

`FAISS(Facevook AI Similarity Search)`  
&nbsp;&nbsp;&nbsp;&nbsp;▶ 효율적인 벡터 유사도 검색 및 클러스터링을 위한 오픈소스 벡터 저장소  
&nbsp;&nbsp;&nbsp;&nbsp;▶ faiss-cpu 라이브러리로 설치 가능(faiss-gpu)  
&nbsp;&nbsp;&nbsp;&nbsp;▶ 주요 특징  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 대규모 벡터 세트에서 효율적인 검색  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 대용량 데이터셋 처리(RAM 효율적 활용)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- GPU 가속 지원  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 다양한 인덱싱 알고리즘 제공(속도와 정확도 조절 가능)

텍스트 벡터를 저장하고, 질문이 들어오면 가장 유사한 문서를 빠르게 찾아주는 '검색 엔진' 역할을 하는 Chroma 벡터 데이터베이스를 사용하기 위해 import 한다.

In [25]:
from langchain_chroma import Chroma

앞서 자른 텍스트 조각(chunk)들을 임베딩 모델을 이용해서 숫자로 바꾸고, 이를 Chroma 벡터 데이터베이스로 로컬(디스크) 저장한다.

In [26]:
# from_documents() 메소드로 청크, 임베딩 모델, 테이블 이름, 테이블이 저장될 경로를 넘겨서 Chroma 데이터베이스를 만든다.
vectorstore = Chroma.from_documents(
    documents=texts, # CharacterTextSplitter로 만든 Chroma 데이터베이스에 저장할 청크를 지정한다. 청크(Document 객체)를 저장한 리스트를 지정해야 한다.
    embedding=embeddings, # 텍스트를 벡터로 변환할 때 사용한 OpenAIEmbeddings로 만든 임베딩 모델을 지정한다.
    collection_name='chroma_test', # 생성하는 Chroma 데이터베이스 내부의 테이블(벡터 저장소) 이름을 지정한다.
    persist_directory='./chroma_db', # 데이터를 메모리에만 두지 않고, 물리적인 파일로 저장할 디렉토리(폴더)를 지정한다.
)

In [27]:
# _collection 속성으로 테이블(벡터 저장소) 이름을 얻어올 수 있고 count()로 저장된 청크의 개수를 얻어올 수 있다.
print(f'벡터 저장소에 이름: {vectorstore._collection}')
print(f'벡터 저장소에 저장된 문서 개수: {vectorstore._collection.count()}')

벡터 저장소에 이름: Collection(name=chroma_test)
벡터 저장소에 저장된 문서 개수: 5


벡터 저장소(vectorstore)에 저장된 수많은 청크들 중, 질문과 가장 유사한 의미를 가진 청크를 찾아온다.  

작동 방식  
질문을 벡터화 한다. => 저장된 모든 청크들과의 유사도를 계산한다. => 가장 가까운 4개(기본값)의 문서를 얻어온다.  
단어가 정확히 일치하지 않아도 의미가 비슷하면 찾아낸다.(예: '설립자' vs '창업자')

In [28]:
query = '테슬라 설립자는 누구인가요?'
# 벡터 스토어에서 similarity_search() 메소드의 인수로 쿼리를 넘겨서 검색을 할 수 있다.
result = vectorstore.similarity_search(query)

In [29]:
# similarity_search() 메소드의 검색할 청크 개수(k)의 기본값은 4이므로, 보통 4가 출력된다.
print(f'검색된 청크 개수: {len(result)}')
# result[0]: 검색된 4개의 문서 중 유사도가 가장 높은(가장 정답에 가까운) 첫 번째 청크를 의미한다.
print(f'검색 결과의 첫 번째 청크: {result[0].page_content[:100]}')

검색된 청크 개수: 4
검색 결과의 첫 번째 청크: 테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는


## 검색기(Retriever)

질의에 관련된 문서를 검색하는 컴포넌트

앞서 구축한 로컬 저장소(vectorstore)를 `검색기(retriever)` 인터페이스로 변환하여, 질문에 대한 답을 찾는 과정을 더 정밀하게 제어한다.

In [37]:
# as_retriever() 메소드로 단순 저장소인 vectorstore를 질문 던지면 답을 찾아오는 '검색 전용 도구'로 변환한다.
# langchain의 다른 구성 요소와 연결할 때 필수적이다.
# search_kwargs 속성에 딕셔너리 형태로 검색기가 검색해서 가져올 청크의 개수를 지정한다. k의 기본값은 similarity_search()와 같은 4개 이다.
# 관련성이 매우 높은 핵심 정보만 추출하고 싶거나, LLM에 전달할 토큰(비용)을 절약하고 싶을 때 사용한다.
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

# invoke() 메소드의 인수로 쿼리를 넘겨서 langchain 표준 방식(Runnable 인터페이스)으로 검색을 실행한다.
# 입력된 쿼리('테슬라 설립자는 누구인가요?')와 가장 의미가 가까운 청크 2개를 찾아서 가져온다.
retriever_docs = retriever.invoke(query)

# 실제로 몇 개의 청크가 검색되었는지 확인한다. 위에서 k를 2로 설정했으므로, 검색 결과가 존재한다면 2가 출력된다.
print(f'검색된 청크 개수: {len(retriever_docs)}')
print(f'첫 번째 관련 청크 내용 보기: {retriever_docs[0].page_content[:100]}')

검색된 청크 개수: 2
첫 번째 관련 청크 내용 보기: 테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는


## 언어 모델(LLMs)

텍스트 생성을 담당하는 대규모 언어 모델로 OpenAI의 GPT 모델 외에도 다양한 모델을 지원한다.
<br><br>
<img src="./LLMs.png" width="1300" align="left" />


context(외부 지식)을 제공하지 않고 답변을 생성한다. 환각(hallucination) 현상이 발생될 수 있다.

`환각 현상`은 AI 모델이 훈련 데이터나 사실에 기반하지 않은 잘못된 정보를 마치 진실인 것처럼 그럴듯하게 만들어내는 현상을 뜻한다. 주로 생성형 AI가 학습 데이터 부족, 잘못된 가정, 혹은 정보의 편향 등으로 인해 `모르는 것을 아는 것처럼 지어내는` 오류를 말한다.

RAG 시스템의 마지막 단계인 `답변 생성(generation)`을 담당하는 거대 언어 모델(LLM)을 사용하기 위해서 ChatOpenAI를 import 한다.  
텍스트를 입력받아 자연스러운 문장으로 응답을 생성하는 인공지능 모델에 연결할 준비를 한다.

In [38]:
from langchain_openai import ChatOpenAI

In [60]:
llm = ChatOpenAI(
    # 사용할 GPT 모델을 지정한다.
    # gpt-4o 모델의 경량화 버전으로, 속도가 빠르고 비용이 저렴하면서도 RAG 답변 생성에는 충분히 똑똑한 성능을 보여준다.
    model='gpt-4o-mini',
    # 답변의 '무작위성(창의성)'을 조정한다. 0.0 ~ 2.0 사이의 값을 지정한다.
    # 0으로 설정한 이유는 RAG 시스템이 마음대로 말을 지어내지 않고, 검색된 청크에 근거하여 가장 정확하고 일관된 답변을 내놓아아 하기 때문이다.
    # 자연스러운 답변을 받아봐야 할 경우에는 보통 0.7 정도를 지정하고 RAG 시스템에서는 보통 0으로 고정한다.
    temperature=0,
    # 생성할 답변의 최대 길이를 제한한다. 답변이 너무 길어져서 비용이 많이 나오거나 출력이 잘리는 것을 방지한다.
    max_completion_tokens=100 # 기존 방식대로 max_tokens=100 형태로 지정해도 된다.
)

# LLM에 invoke() 메소드의 인수로 쿼리를 던져서 실행한다.
# 입력한 질문이 OpenAI 서버로 전송되고, GPT 모델이 학습한 지식을 바탕으로 답변을 생성해서 다시 보내준다.
# 현재 코드는 '검색 결고'를 전달하지 않고 GPT가 원래 알고 있던 지식으로만 답변하는 상태이다. 순수 LLM 호출이다.
response = llm.invoke('테슬라 설립자는 누구인가요?')
response.content

'테슬라의 설립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, CEO로 취임하며 회사의 방향성을 크게 변화시켰습니다. 이후 그는 테슬라의 가장 유명한 얼굴이'

문맥(context, 추가정보)을 제공하고 답변 생성

RAG 핵심 과정인 `문맥 주입(context injection)`은 검색된 문서와 사용자의 질문을 하나로 합쳐서 모델이 `자신의 지식이 아닌, 제공된 문서 안에서만` 답하도록 유도하는 프롬프트를 만드는 과정이다.

In [59]:
# 파이썬의 f-string(포맷 스트링)과 여러 줄 문자열(따옴표 3개)을 사용하여 LLM에 보낼 프롬프트를 만든다.
query_context = f'''
{retriever_docs[0].page_content}\n
위 내용에 근거하여 다음 질문에 답변해주세요.\n
{query}
'''
print(query_context)


테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.

위 내용에 근거하여 다음 질문에 답변해주세요.

테슬라 설립자는 누구인가요?



In [61]:
response = llm.invoke(query_context)
response.content

'테슬라의 설립자는 마틴 에버하드와 마크 타페닝입니다.'

# RAG 파이프라인 구현

<img src="./pipeline.png" width="700" align="left" />

LLM에게 전달할 메시지 형식(prompt)를 정의하기 위해 ChatOpenAI를 사용중이므로 ChatPromptTemplate를 import 한다.

In [62]:
from langchain_core.prompts import ChatPromptTemplate

검색된 여러 문서를 하나의 컨텍스트로 결합해서 프롬프트에 통째로 넣고(stuff) 답변하는 체인을 생성하기 위해 `create_stuff_documents_chain`를 import 한다.  
검색기로 질문에 관련된 문서를 검색하고, 최종 답변을 생성하는 RAG 파이프라인을 구축하기 위해 `create_retrieval_chain`를 import 한다.

LangChain 1.0 버전부터 `langchain.chains` 모듈이 `langchain_classic.chains` 모듈로 이동되었다.

In [64]:
# from langchain.chains.combine_documents import create_stuff_documents_chain # LangChain 1.0 버전부터 사용할 수 없다.
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# from langchain.chains import create_retrieval_chain # LangChain 1.0 버전부터 사용할 수 없다.
from langchain_classic.chains import create_retrieval_chain

In [78]:
# LLM이 어떻게 행동해야 할지 지시하는 프롬프트 템플릿을 정의한다.
# from_template() 메소드에 프롬프트로 사용할 문자열을 넘겨서 프롬프를 만든다.
# {context}는 검색기에서 찾아온 문서 내용이 들어갈 자리이고, {input}은 사용자가 던진 실제 질문이 들어갈 자리이다.
prompt = ChatPromptTemplate.from_template('''
다음 컨텍스트를 바탕으로 질문에 답변해주세요.
컨텍스트에 관련된 정보가 없다면, '주어진 정보로는 답변할 수 없습니다.'라고 말씀해주세요.

컨텍스트:
{context}

질문:
{input}

답변:
''')


# LLM 모델을 정의한다.
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# create_stuff_documents_chain와 create_retrieval_chain를 사용해서 chain을 생성하고 연결한다.
# '검색된 문서들을 프롬프트에 넣고 답변을 만들어라'라는 문서 처리 단계를 정의한다.
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
# 사용자의 질문을 바탕으로 검색기가 관련 문서를 찾아 찾은 문서들을 combine_docs_chain에 전달하는 최종 답변을 생성하는 전체 RAG 프로세스를 완성한다.
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# RAG chain을 실행한다.
query = '테슬라 설립자는 누구인가요?'
# 내부적으로 문서 검색 => 검색된 문서 결합 => 프롬프트에 검색된 문서 주입 => LLM 답변
response = rag_chain.invoke({'input': query})

In [79]:
response

{'input': '테슬라 설립자는 누구인가요?',
 'context': [Document(id='57d51b60-fd96-42a2-8545-dd439419bedf', metadata={'source': './data\\테슬라_KR.txt'}, page_content='테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.'),
  Document(id='ce70895c-f1e1-4a18-b907-b070ea3cdda8', metadata={'source': './data\\리비안_KR.txt'}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다. 2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다. 주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다.')],
 'answer': '테슬라의 설립자는 마틴 에버하드와 마크 타페닝입니다.'}

In [80]:
response.keys()

dict_keys(['input', 'context', 'answer'])

In [82]:
response.get('answer')

'테슬라의 설립자는 마틴 에버하드와 마크 타페닝입니다.'

In [84]:
response['answer']

'테슬라의 설립자는 마틴 에버하드와 마크 타페닝입니다.'

## create_stuff_documents_chain 추가 설명

검색된 문서들을 하나의 컨텍스트로 결합하고, 이를 바탕으로 질문에 답변하는 체인을 생성한다.

LangChain에서 사용하는 표준 데이터 형식인 Document를 사용하기 위해 import 한다.

In [85]:
from langchain_core.documents import Document

In [91]:
# 프롬프트에 넣어줄 Document 객체가 리스트에 저장된 샘플 문서를 만든다.
# 실제 RAG 시스템에서는 벡터 저장소에서 자동으로 찾아오겠지만, 여기서는 테스트를 위해 두 개의 문서(Document 객체)를 수동으로 정의한다.
docs = [
    Document(page_content='LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.'),
    Document(page_content='LangChain은 파이썬과 자바스크립트 및 자바를 지원합니다.'),
]

# 문서 결합 체인을 생성한다.
# stuff(채우다) 방식의 체인을 생성한다.
# 프롬프트의 {context} 위치에 리스트로 전달되는 여러 개의 문서를 모두 넣어주고 LLM에게 전달하는 로직을 준비한다.
stuff_chain = create_stuff_documents_chain(llm, prompt)

# 체인 실행
# 준비된 체인에 질문(input)과 추가정보(context)를 전달해서 실행한다.
# docs 리스트 안에 있는 내용들을 하나로 합쳐진다. => 합쳐진 내용을 프롬프트의 {context}에 넣는다. => 프롬프트의 {input}에 질문을 넣는다. => 완성된 최종
# 프롬프트를 LLM에게 보내서 답변을 받는다.
response = stuff_chain.invoke({
    'context': docs,
    'input': 'LangChain은 어떤 언어를 지원하나요?'
})

In [92]:
response

'LangChain은 파이썬, 자바스크립트 및 자바를 지원합니다.'

## create_retrieval_chain 추가 설명

질문에 관련된 문서를 검색하고, 검색된 문서를 바탕으로 답변을 생성하는 전체 RAG 파이프라인을 구축한다.

위 코드의 문서 결합 체인(stuff_chain)에 검색기를 연결하여, 질문과 관련된 내용만 찾아와서 답변하게 만드는 RAG 파이프라인을 구축한다.

In [94]:
# 검색기 체인을 생성한다.
# create_retrieval_chain() 함수의 첫 번째 인수로 지정된 검색기가 찾아온 문서들을 두 번째 인수인 stuff_chain의 프롬프트에 넣고 RAG 체인을 만든다.
retrieval_chain = create_retrieval_chain(retriever, stuff_chain)

query = '테슬라 설립자는 누구인가요?'
# 전체 RAG 프로세스를 가동한다.
# 검색(retrieval): '테슬라 설립자'라는 키워드로 벡터 스토어에서 관련 문서를 찾아낸다.
# 전달(context): 찾아낸 문서를 context라는 이름으로 stuff_chain의 프롬프에 전달한다.
# 답변(generate): LLM이 전달받은 문서들을 참고하여 질문에 대한 최종 답변을 생성한다.
response = retrieval_chain.invoke({'input': query})

In [96]:
response['answer']

'테슬라의 설립자는 마틴 에버하드와 마크 타페닝입니다.'

# Gradio 챗봇 인터페이스 구현

Gradio 챗봇 인터페이스를 사용하기 위해 gradio를 import 한다.

In [97]:
import gradio as gr

Gradio 라이브러리를 사용해서 LangChain 기반의 RAG 시스템을 웹 기반의 챗봇 인터페이스로 구축하고 실행한다.

챗봇 응답 처리 함수를 정의한다.  
`message`: 사용자가 챗봇 입력창에 새롭게 입력한 문자열  
`history`: 이전까지 주고받은 대화 내역이 리스트 형태로 자동 전달된다.

In [99]:
def answer_invoke(message, history):
    response = rag_chain.invoke({'input': message})
    return response['answer']

Gradio 챗봇 인터페이스를 생성한다.  
`fn`: 사용자가 메시지를 전송할 때 실행할 함수를 지정한다.  
`title`: 웹 UI 상단에 표시될 제목을 지정한다.

ChatInterface() 메소드를 실행하면 입력창, send 버튼 등이 별도의 UI 코딩 없이 자동으로 생성된다.

In [100]:
chatbot = gr.ChatInterface(fn=answer_invoke, title='QA Bot')

Gradio 챗봇을 실행한다.

쥬피터 노트북 또는 코랩 환경에서 실행할 경우 셀 출력창에 챗봇 화면이 바로 렌더링된다.  
실행 시 로컬 주소가(`http://127.0.0.1:7860`) 출력되며 해당 링크를 ctrl키를 누른채 클릭하거나 복사해서 브라우저의 주소창에 붙어넣어도 된다.

In [103]:
chatbot.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Gradio 종료

In [102]:
chatbot.close()

Closing server running on port: 7860
